In [0]:
file_path = "/Volumes/data3/default/pd5201_klastry/SRR16356247_1.fastq"

lines_df = spark.read.text(file_path)

lines_df.show(8, truncate=False)

print(f"Liczba linii: {lines_df.count()}")
print(f"Liczba odczytów: {lines_df.count() // 4}")

+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|@SRR16356247.1 1 length=151                                                                                                                            |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|
|+SRR16356247.1 1 length=151                                                                                                                            |
|A=AFFFFFFFFFFFFFF#F/FAFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF#FFFFFFFFFF#FFFFF

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id

window = Window.orderBy(monotonically_increasing_id())

lines_with_id = lines_df.withColumn(
    "line_number",
    row_number().over(window) - 1
)

lines_with_id.show(8, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|value                                                                                                                                                  |line_number|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |
|+SRR16356247.1 1 length=151                                                                                                                            |2          |
|A=A

In [0]:
from pyspark.sql.functions import when, col

lines_typed = lines_with_id.withColumn(
    "line_type",
    when(col("line_number") % 4 == 0, "header")
    .when(col("line_number") % 4 == 1, "sequence")
    .when(col("line_number") % 4 == 2, "separator")
    .when(col("line_number") % 4 == 3, "quality")
)

lines_typed.show(12, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|value                                                                                                                                                  |line_number|line_type|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |
|+SRR16356247.1 1 length=151                                                                                            

In [0]:
lines_with_record = lines_typed.withColumn(
    "record_id",
    (col("line_number") / 4).cast("integer")
)

lines_with_record.show(16, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|value                                                                                                                                                  |line_number|line_type|record_id|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |0        |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |0        |
|+SRR16356247.1 1 length=151                                          

In [0]:
from pyspark.sql.functions import first

fastq_wide = lines_with_record.groupBy("record_id").pivot("line_type").agg(
    first("value")
)

fastq_wide.show(5, truncate=False)
fastq_wide.printSchema()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|header                     |quality                                                                                                                                                |separator                  |sequence                                                                                                                                               |
+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------

In [0]:
from pyspark.sql.functions import col, regexp_replace, split

fastq_clean = fastq_wide.withColumn(
    "read_id",
    split(
        regexp_replace(col("header"), "^@", ""),
        " "
    )[0]
)

fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
).show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|read_id      |sequence                                                                                                                                               |quality                                                                                                                                                |
+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|0        

In [0]:
fastq_clean.printSchema()

root
 |-- record_id: integer (nullable = true)
 |-- header: string (nullable = true)
 |-- quality: string (nullable = true)
 |-- separator: string (nullable = true)
 |-- sequence: string (nullable = true)
 |-- read_id: string (nullable = true)



In [0]:
fastq_clean = fastq_clean.select(
    "record_id",
    "read_id",
    "header",
    "sequence",
    "quality"
)

fastq_clean.show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|read_id      |header                     |sequence                                                                                                                                               |quality                                                                                                                                                |
+---------+-------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------

In [0]:
from pyspark.sql.functions import regexp_extract, length, col

fastq_validation = (
    fastq_clean
    .withColumn(
        "declared_length",
        regexp_extract(col("header"), r"length=(\d+)", 1).cast("int")
    )
    .withColumn(
        "actual_length",
        length(col("sequence"))
    )
)

fastq_validation.select(
    "read_id",
    "declared_length",
    "actual_length"
).show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------+---------------+-------------+
|read_id      |declared_length|actual_length|
+-------------+---------------+-------------+
|SRR16356247.1|151            |151          |
|SRR16356247.2|151            |151          |
|SRR16356247.3|151            |151          |
|SRR16356247.4|151            |151          |
|SRR16356247.5|151            |151          |
+-------------+---------------+-------------+
only showing top 5 rows


In [0]:
mismatched_reads = fastq_validation.filter(
    col("declared_length") != col("actual_length")
)

print(f"Liczba rekordów z niezgodną długością: {mismatched_reads.count()}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Liczba rekordów z niezgodną długością: 0


Analiza wykonania zadania 1

-Ile Jobs zostało utworzonych? Czy widzimy zależność między liczbą wywołań akcji a liczbą zadań?
 
 W tym zadaniu zostało utworzone 1 Job

-Które operacje były transformacjami, które akcjami?

Transformacje:
- withColumn()
- regexp_extract()
- length()
- filter()

Akcja:
- count()

-Sprawdźmy DAG Visualization dla tego Joba. Ile etapów (RDD/DataFrame) tworzy plan wykonania zapytania?

widocznych jest około 10 operatorów planu wykonania (Scan, Row To Column, Shuffle, Sort, Window, Grouping Aggregate, Filter, Aggregate, Column To Row), natomiast liczba Stage nie jest bezpośrednio prezentowana przez interfejs Serverless.

Ile tasków wykonał jedyny Stage w tym Jobie? Porównajmy tę liczbę z domyślną liczbą partycji DataFrame. Wyciągnijmy wniosek o relacji między partycjami a taskami.

Na podstawie Query Profile wykonano 18 tasków. W Spark liczba tasków w Stage zwykle odpowiada liczbie partycji przetwarzanego DataFrame, dlatego można wnioskować, że dane zostały podzielone na około 18 partycji.

In [0]:
from pyspark.sql.functions import col

low_quality_reads = fastq_clean.filter(
    col("quality").contains("#")
)

low_quality_reads.show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|read_id      |header                     |sequence                                                                                                                                               |quality                                                                                                                                                |
+---------+-------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------

In [0]:
print(f"Liczba odczytów zawierających znak #: {low_quality_reads.count()}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Liczba odczytów zawierających znak #: 5684


● Porównajmy czas wykonania tego zadania z Zadaniem 1. Które było szybsze i dlaczego? (podpowiedź: porównajmy złożoność operacji regexp_extract i length z prostym contains).

Czas wykonania zadania 2 był krótszy niż zadania 1. W Zadaniu 1 wykonano regexp_extract() do wyciągnięcia długości z nagłówków oraz length() do obliczenia długości sekwencji. W Zadaniu 2 wykorzystano jedynie prostą operację contains("#"), która sprawdza obecność pojedynczego znaku. Jest to operacja mniej kosztowna obliczeniowo, dlatego czas wykonania był krótszy.

● Czy Spark wykorzystał cache? Przyjrzyjmy się liście stage'ów. Czy któryś z nich ma przy sobie zieloną etykietę "skipped"? Jeśli tak, oznacza to, że Spark nie musiał ponownie czytać danych z dysku, ponieważ skorzystał z wyniku zapisanego w pamięci przez .cache() w poprzednim kroku.

Nie. W zadaniu 2 nie zastosowano metody cache(), dlatego Spark ponownie wykonał wszystkie operacje. W środowisku Serverless nie widać etapów oznaczonych jako "skipped", dlatego nie można tego zweryfikować tak jak w klasycznym Spark UI..

● Jaki był Locality Level tasków? W szczegółach Stage'a znajdźmy informację o "Locality Level". Czy dominowało PROCESS_LOCAL (najlepsza opcja, dane w pamięci tego samego procesora)? Jeśli tak, dlaczego?

Środowisko Azure Databricks Serverless nie udostępnia szczegółowych informacji o Locality Level w taki sposób jak klasyczny Spark UI. Odpowiadając teoterycznie dominował PROCESS_LOCAL który jest najkorzystnieszym ponieważ dane są przetwarzane przez ten sam proces, zapewniając najwyższą wydajność.

● Ile rekordów (Records) przetworzył każdy Task? Czy liczba ta jest zgodna z Twoimi oczekiwaniami dotyczącymi podziału danych na partycje?

w zakładce "see performance" widać jedynie wykonanych 9/9 tasków. Serverless nie pokazuje ile rekordów przetworzył każdy task.

In [0]:
from pyspark.sql.functions import col, length

analysis_df = (
    fastq_clean
    .withColumn(
        "has_low_quality",
        col("quality").contains("#")
    )
    .withColumn(
        "seq_length",
        length(col("sequence"))
    )
    .groupBy(
        "has_low_quality",
        "seq_length"
    )
    .count()
    .orderBy(
        "has_low_quality",
        "seq_length"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
analysis_df.cache() # tu wyskakuje błąd prawdopodobnie z powodu ograniczenia serverless

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5671209099678425>, line 1
----> 1 analysis_df.cache()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2133, in DataFrame.cache(self)
   2132 def cache(self) -> ParentDataFrame:
-> 2133     return self.persist()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2140, in DataFrame.persist(self, storageLevel)
   2135 def persist(
   2136     self,
   2137     storageLevel: StorageLevel = (StorageLevel.MEMORY_AND_DISK_DESER),
   2138 ) -> ParentDataFrame:
   2139     relation = self._plan.plan(self._session.client)
-> 2140     self._session.client._analyze(
   2141         method="persist", relation=relation, storage_level=storageLevel
   2142     )
   2143     return self

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client

In [0]:
analysis_df.count()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


2

In [0]:
analysis_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------------+----------+-------+
|has_low_quality|seq_length|  count|
+---------------+----------+-------+
|          false|       151|2313942|
|           true|       151|   5684|
+---------------+----------+-------+



In [0]:
analysis_df.select("seq_length").distinct().count()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


1

In [0]:
analysis_df.printSchema()

root
 |-- has_low_quality: boolean (nullable = true)
 |-- seq_length: integer (nullable = true)
 |-- count: long (nullable = false)



● Ile Jobs zostało utworzonych? Wypiszmy każdą akcję z kodu i przypisz jej numer Joba z UI.

Wykonano 4 akcje:

analysis_df.count()
analysis_df.show()
analysis_df.select("seq_length").distinct().count()
analysis_df.printSchema()

Każda akcja uruchomiła osobny Job.

● Narysujmy schemat Stage'ów dla pierwszego Joba (tego z groupBy):

○ Stage 1: Jakie operacje się tu odbywały? Czy widzimy w metrykach Shuffle Write? (to etap, na którym taski przygotowują dane do agregacji).

w stage 1 wykonywany był odczyt danych, dodanie nowych kolumn (withColumn) oraz grupowanie (groupBy). W środowisku Azure Databricks Serverless nie są wyświetlane metryki Shuffle Write, dlatego nie było możliwości ich sprawdzenia.

○ Stage 2: Jakie operacje? Czy widzimy Shuffle Read? (to etap agregacji, na którym taski czytają dane przygotowane przez Stage 1).

w stage 2 wykonywana była agregacja (count()) oraz sortowanie wyniku (orderBy). W środowisku Azure Databricks Serverless nie są wyświetlane metryki Shuffle Read, dlatego nie było możliwości ich sprawdzenia.

● Które Jobs wykorzystały cache? Zidentyfikujmy Joby, których etapy zostały pominięte ("skipped"). Dlaczego tak się stało?

W Serverless cache był nieudany dlatego ten krok został pominięty.

● Porównajmy Input Size między różnymi Jobami. Dlaczego Jobs korzystające z cache mają bardzo mały lub zerowy rozmiar wejściowy?

Nie było możliwości porównania Input Size, ponieważ środowisko Azure Databricks Serverless nie obsługuje cache(). Teoretycznie Joby korzystające z cache mają bardzo mały lub zerowy Input Size, ponieważ dane są odczytywane z pamięci, a nie ponownie z dysku.

● Efektywność cache: porównajmy czas wykonania pierwszego Joba (który musiał obliczyć wszystko od zera) z czasami kolejnych Jobów (które czytały z cache). Jaki jest zysk?

W Serverless cache był nieudany dlatego ten krok został pominięty. z tego względu nie mogę ocenić szybkoścki kolejnych zadań (odpowiedając teoretycznie pierwszy job wykonywał sioę znacznie dłużej niż kolejne które korzystały z cache)

● Locality Level: Czy wszystkie taski w Jobach korzystających z cache miały PROCESS_LOCAL? Jeśli nie, spróbujmy określić, dlaczego niektóre taski musiały czytać dane z pamięci na innym węźle (NODE_LOCAL)

Nie można było tego sprawdzić, ponieważ środowisko Azure Databricks Serverless nie wyświetla informacji o Locality Level. Teoretycznie po zastosowaniu cache() najbardziej pożądanym poziomem jest PROCESS_LOCAL, ponieważ dane są odczytywane z pamięci lokalnej.